In [1]:
import pandas as pd

path = "/iskra-project/data/texts_with_metadata_20260401.csv"

df = pd.read_csv(path)

print(df.head())
print(df.columns)

                                                path author_folder  \
0  /Users/anastasiabogdanova/R_directory/iskra-pr...         dubia   
1  /Users/anastasiabogdanova/R_directory/iskra-pr...         dubia   
2  /Users/anastasiabogdanova/R_directory/iskra-pr...         dubia   
3  /Users/anastasiabogdanova/R_directory/iskra-pr...         dubia   
4  /Users/anastasiabogdanova/R_directory/iskra-pr...         dubia   

                             file_name  \
0            dubia_finans_manifest.txt   
1        dubia_nasushnie_zadachi_I.txt   
2           dubia_novoe_poboishe_I.txt   
3             dubia_ot_red_iskry_I.txt   
4  dubia_ot_red_na_pismo_parvusa_I.txt   

                                                text  n_chars  n_words  
0  Финансовый манифест.\n\nПравительство на краю ...     4600      584  
1  Насущные задачи нашего движения.\n\nРусская со...    11478     1485  
2  НОВОЕ ПОБОИЩЕ. \n\nПовидимому, мы переживаем м...    11186     1564  
3  ЗАЯВЛЕНИЕ РЕДАКЦИИ «ИСКРЫ».\nОТ

In [3]:
pip install spacy

Note: you may need to restart the kernel to use updated packages.


In [5]:
!python -m spacy download ru_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 14.7 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')


In [7]:
import spacy

nlp = spacy.load("ru_core_news_sm")
print("модель загрузилась")

модель загрузилась


In [9]:
# прогоняем ОДИН текст через модель
text = df["text"].iloc[2]

In [11]:
doc = nlp(text)

In [13]:
# смотрим, как spaCy размечает части речи
for token in doc[:20]:
    print(token.text, token.pos_)

НОВОЕ NOUN
ПОБОИЩЕ PROPN
. PUNCT


 SPACE
Повидимому PROPN
, PUNCT
мы PRON
переживаем VERB
момент NOUN
, PUNCT
когда ADV
наше DET
рабочее ADJ
движеніе NOUN
опять ADV
с ADP
неудержимой ADJ
силой NOUN
приводит VERB
к ADP


In [15]:
# быстрая проверка распределения POS
pos_counts = {}

for token in doc:
    if not token.is_punct:
        pos = token.pos_
        pos_counts[pos] = pos_counts.get(pos, 0) + 1

print(pos_counts)

{'NOUN': 489, 'PROPN': 11, 'SPACE': 12, 'PRON': 75, 'VERB': 221, 'ADV': 87, 'DET': 62, 'ADJ': 205, 'ADP': 154, 'CCONJ': 90, 'PART': 85, 'SCONJ': 37, 'NUM': 20, 'AUX': 17, 'PUNCT': 1}


In [17]:
# оформляем в функцию
def get_pos_counts(text):
    doc = nlp(text)
    counts = {}
    
    for token in doc:
        if not token.is_punct:
            pos = token.pos_
            counts[pos] = counts.get(pos, 0) + 1
    
    return counts

In [19]:
# проверяем функцию на одном тексте
pos_counts = get_pos_counts(df["text"].iloc[0])

print(pos_counts)

{'ADJ': 100, 'NOUN': 203, 'SPACE': 22, 'ADP': 60, 'PRON': 22, 'VERB': 87, 'CCONJ': 32, 'PART': 14, 'ADV': 23, 'DET': 19, 'NUM': 5, 'SCONJ': 6, 'AUX': 1, 'PROPN': 12}


In [21]:
# нормализация
def normalize_pos(counts):
    total = sum(counts.values())
    
    if total == 0:
        return counts
    
    return {k: v / total for k, v in counts.items()}

In [23]:
# проверка
norm = normalize_pos(pos_counts)
print(norm)

{'ADJ': 0.16501650165016502, 'NOUN': 0.334983498349835, 'SPACE': 0.036303630363036306, 'ADP': 0.09900990099009901, 'PRON': 0.036303630363036306, 'VERB': 0.14356435643564355, 'CCONJ': 0.052805280528052806, 'PART': 0.0231023102310231, 'ADV': 0.037953795379537955, 'DET': 0.03135313531353135, 'NUM': 0.00825082508250825, 'SCONJ': 0.009900990099009901, 'AUX': 0.0016501650165016502, 'PROPN': 0.019801980198019802}


In [27]:
# ВАРИАНТ 1. Убираем DUBIA-тексты
train_df = df[df["author_folder"] != "dubia"]

In [29]:
# ВАРИАНТ 1. Применяем функцию с прогресс-баром

from tqdm import tqdm
tqdm.pandas()

train_df = df[df["author_folder"] != "dubia"].copy() # делаем копию
train_df["pos_profile"] = train_df["text"].progress_apply(get_pos_counts)

100%|█████████████████████████████████████████| 145/145 [03:49<00:00,  1.58s/it]


In [185]:
# ВАРИАНТ 2. для distance-based methods

from tqdm import tqdm
tqdm.pandas()

train_df = df.copy()
train_df["pos_profile"] = train_df["text"].progress_apply(get_pos_counts)

100%|█████████████████████████████████████████| 154/154 [03:48<00:00,  1.48s/it]


In [187]:
# Сбрасываем индексы после фильтрации
train_df = train_df.reset_index(drop=True)

# Нормализуем POS
train_df["pos_profile"] = train_df["pos_profile"].progress_apply(normalize_pos)

100%|███████████████████████████████████████| 154/154 [00:00<00:00, 2661.83it/s]


In [189]:
# Развернуть в матрицу
pos_df = pd.json_normalize(train_df["pos_profile"]).fillna(0)

# Сбрасываем индексы pos_df на всякий случай
pos_df = pos_df.reset_index(drop=True)

In [191]:
# Собрать итог, поменяла колонки местами, чтобы были как в MFW
feature_matrix = pd.concat(
    [
        train_df[["file_name"]].reset_index(drop=True),
        pos_df,
        train_df[["author_folder"]].reset_index(drop=True)
    ],
    axis=1
)

In [37]:
# Сохранить
feature_matrix.to_csv("/Users/anastasiabogdanova/R_directory/iskra-project/features/pos_18_nodubia.csv", index=False)

In [193]:
# ВАРИАНТ 2. Сохранить
feature_matrix.to_csv("/Users/anastasiabogdanova/R_directory/iskra-project/features/pos_18.csv", index=False)

In [ ]:
Добавляем ДОПОЛНИТЕЛЬНЫЕ ПРИЗНАКИ к базовым 18 POS
1. 2-граммы POS (последовательности двух тегов)
Например: NOUN + VERB, ADJ + NOUN, PRON + VERB
Каждая такая последовательность = новый признак
Показывает типичные структуры предложений автора, а не отдельные слова
2. 3-граммы POS (тройки тегов)
Например: DET + ADJ + NOUN, NOUN + VERB + NOUN
Даёт более детальное «стилометрическое отпечатание»
3. Синтаксические паттерны (опционально, чуть сложнее)
Например: количество подчинённых предложений, количество вводных конструкций, частота прямого/косвенного падежа и т.д.
Это можно взять из зависимости doc[i].dep_ в spacy

In [ ]:
import pandas as pd
import spacy
from tqdm import tqdm

df = pd.read_csv("/Users/anastasiabogdanova/R_directory/iskra-project/data/texts_with_metadata_20260401.csv")

nlp = spacy.load("ru_core_news_sm")
tqdm.pandas()

def get_pos_ngrams(text, n=2):
    doc = nlp(text)
    pos_tags = [token.pos_ for token in doc]
    ngrams = zip(*[pos_tags[i:] for i in range(n)])
    ngram_counts = {}
    for ng in ngrams:
        key = "_".join(ng)
        ngram_counts[key] = ngram_counts.get(key, 0) + 1
    total = sum(ngram_counts.values())
    # нормализация по числу n-грамм
    for k in ngram_counts:
        ngram_counts[k] /= total
    return ngram_counts

# пример для всех текстов
train_df = df[df["author_folder"].str.lower() != "dubia"].copy()
train_df.reset_index(drop=True, inplace=True)
# создаём колонки с n-gram
train_df["pos_2grams"] = train_df["text"].progress_apply(lambda x: get_pos_ngrams(x, n=2))
train_df["pos_3grams"] = train_df["text"].progress_apply(lambda x: get_pos_ngrams(x, n=3))

In [41]:
# === Превращаем n-граммы в DataFrame ===
pos_2_df = pd.DataFrame(train_df["pos_2grams"].to_list()).fillna(0)
pos_3_df = pd.DataFrame(train_df["pos_3grams"].to_list()).fillna(0)


In [51]:
# Добавляю префиксы к именам колонок
pos_2_df.columns = ["pos2_" + col for col in pos_2_df.columns]
pos_3_df.columns = ["pos3_" + col for col in pos_3_df.columns]

In [75]:
# Убрать редкие признаки
pos_2_df = pos_2_df.loc[:, (pos_2_df > 0).sum() > 5]
pos_3_df = pos_3_df.loc[:, (pos_3_df > 0).sum() > 10]

In [91]:
# Проверка на синхронность строк между таблицами
len(train_df) == len(pos_2_df) == len(pos_3_df)

True

In [ ]:
ПРИВОДИМ матрицы К ЕДИНОМУ С MFW формату

In [83]:
pos_2gram_matrix = pd.concat(
    [
        train_df[["file_name"]].reset_index(drop=True),
        pos_2_df.reset_index(drop=True),
        train_df[["author_folder"]].reset_index(drop=True)
    ],
    axis=1
)

In [85]:
pos_3gram_matrix = pd.concat(
    [
        train_df[["file_name"]].reset_index(drop=True),
        pos_3_df.reset_index(drop=True),
        train_df[["author_folder"]].reset_index(drop=True)
    ],
    axis=1
)

In [87]:
# Сохранить 
pos_2gram_matrix.to_csv("/iskra-project/features/exploratory/pos_2gram.csv", index=False)

print("Матрица POS_2gram сохранена! Размерность:", pos_2gram_matrix.shape)

Матрица POS_2gram сохранена! Размерность: (145, 245)


In [89]:
pos_3gram_matrix.to_csv("/iskra-project/features/exploratory/pos_3gram.csv", index=False)

print("Матрица POS_3gram сохранена! Размерность:", pos_3gram_matrix.shape)

Матрица POS_3gram сохранена! Размерность: (145, 1848)


In [ ]:
# ДЕЛАЕМ МАТРИЦУ Леммы со взвешиванием TF-IDF

In [115]:
# Лемматизация без дубиа
import spacy
import pandas as pd
from tqdm import tqdm

nlp = spacy.load("ru_core_news_sm")
tqdm.pandas()

df = pd.read_csv("/Users/anastasiabogdanova/R_directory/iskra-project/data/texts_with_metadata_20260401.csv")

train_df = df[df["author_folder"].str.lower() != "dubia"].copy()
train_df.reset_index(drop=True, inplace=True)

def lemmatize(text):
    doc = nlp(text)
    lemmas = [token.lemma_.lower() for token in doc 
              if token.is_alpha] # стоп-слова не удалялись
    return lemmas

In [195]:
# ВАРИАНТ 2. Лемматизация
import spacy
import pandas as pd
from tqdm import tqdm

nlp = spacy.load("ru_core_news_sm")
tqdm.pandas()

df = pd.read_csv("/Users/anastasiabogdanova/R_directory/iskra-project/data/texts_with_metadata_20260401.csv")

train_df = df.copy()
train_df.reset_index(drop=True, inplace=True)

def lemmatize(text):
    doc = nlp(text)
    lemmas = [token.lemma_.lower() for token in doc 
              if token.is_alpha] # стоп-слова не удалялись
    return lemmas

In [197]:
# Список лемм
train_df["lemmas"] = train_df["text"].progress_apply(lemmatize)

100%|█████████████████████████████████████████| 154/154 [03:45<00:00,  1.47s/it]


In [199]:
# Частотная матрица (TF-IDF) 3000
from sklearn.feature_extraction.text import TfidfVectorizer

train_df["lemmas_str"] = train_df["lemmas"].apply(lambda x: " ".join(x))

vectorizer = TfidfVectorizer(min_df=5, max_df=0.6, max_features=3000)

X = vectorizer.fit_transform(train_df["lemmas_str"])

lemma_df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out()
)

In [201]:
# Приводим к общему формату матрицы
lemma_matrix = pd.concat(
    [
        train_df[["file_name"]].reset_index(drop=True),
        lemma_df.reset_index(drop=True),
        train_df[["author_folder"]].reset_index(drop=True)
    ],
    axis=1
)

In [145]:
# Сохраняем без дубиа
lemma_matrix.to_csv("/Users/anastasiabogdanova/R_directory/iskra-project/features/lemma_3000.csv", index=False)

print("Частотная матрица сохранена! Размерность:", lemma_matrix.shape)

Частотная матрица сохранена! Размерность: (145, 3002)


In [203]:
# ВАРИАНТ 2. Сохраняем
lemma_matrix.to_csv("/Users/anastasiabogdanova/R_directory/iskra-project/features/lemma_3000_full.csv", index=False)

print("Частотная матрица сохранена! Размерность:", lemma_matrix.shape)

Частотная матрица сохранена! Размерность: (154, 3002)


In [147]:
print(lemma_matrix.head())
print(lemma_matrix.shape)
print(lemma_matrix.columns[:10])

                                file_name  credo   de        ii  iii   iv  \
0         krupskaya_k_voposu_o_shkole.txt    0.0  0.0  0.000000  0.0  0.0   
1            krupskaya_o_staroy_iskre.txt    0.0  0.0  0.021305  0.0  0.0   
2  krupskaya_samoubiystva_uchashihsya.txt    0.0  0.0  0.000000  0.0  0.0   
3          krupskaya_shkolnie_kolonii.txt    0.0  0.0  0.000000  0.0  0.0   
4      krupskaya_sovmestnoe_obuchenie.txt    0.0  0.0  0.000000  0.0  0.0   

    la  sic  xix  абсолютизм  ...  январь  японец  япония  японский     яркий  \
0  0.0  0.0  0.0         0.0  ...     0.0     0.0     0.0       0.0  0.000000   
1  0.0  0.0  0.0         0.0  ...     0.0     0.0     0.0       0.0  0.000000   
2  0.0  0.0  0.0         0.0  ...     0.0     0.0     0.0       0.0  0.013576   
3  0.0  0.0  0.0         0.0  ...     0.0     0.0     0.0       0.0  0.000000   
4  0.0  0.0  0.0         0.0  ...     0.0     0.0     0.0       0.0  0.009708   

   ярко  ясно  ясность     ясный  author_folder  


In [205]:
# Частотная матрица (TF-IDF) 1000
from sklearn.feature_extraction.text import TfidfVectorizer

train_df["lemmas_str"] = train_df["lemmas"].apply(lambda x: " ".join(x))

vectorizer = TfidfVectorizer(min_df=5, max_df=0.6, max_features=1000)

X = vectorizer.fit_transform(train_df["lemmas_str"])

lemma_df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out()
)

In [207]:
# Приводим к общему формату матрицы
lemma_matrix = pd.concat(
    [
        train_df[["file_name"]].reset_index(drop=True),
        lemma_df.reset_index(drop=True),
        train_df[["author_folder"]].reset_index(drop=True)
    ],
    axis=1
)

In [153]:
# Сохраняем без дубиа
lemma_matrix.to_csv("/Users/anastasiabogdanova/R_directory/iskra-project/features/lemma_1000.csv", index=False)

print("Частотная матрица сохранена! Размерность:", lemma_matrix.shape)

Частотная матрица сохранена! Размерность: (145, 1002)


In [209]:
# ВАРИАНТ 2. Сохраняем
lemma_matrix.to_csv("/Users/anastasiabogdanova/R_directory/iskra-project/features/lemma_1000_full.csv", index=False)

print("Частотная матрица сохранена! Размерность:", lemma_matrix.shape)

Частотная матрица сохранена! Размерность: (154, 1002)


In [155]:
print(lemma_matrix.head())
print(lemma_matrix.shape)
print(lemma_matrix.columns[:10])

                                file_name        ii  абсолютизм     автор  \
0         krupskaya_k_voposu_o_shkole.txt  0.000000         0.0  0.000000   
1            krupskaya_o_staroy_iskre.txt  0.023572         0.0  0.000000   
2  krupskaya_samoubiystva_uchashihsya.txt  0.000000         0.0  0.000000   
3          krupskaya_shkolnie_kolonii.txt  0.000000         0.0  0.000000   
4      krupskaya_sovmestnoe_obuchenie.txt  0.000000         0.0  0.008408   

   агитация  аграрный  аксельрод  активный  анализ  анархический  ...  эпоха  \
0       0.0       0.0   0.000000       0.0     0.0           0.0  ...    0.0   
1       0.0       0.0   0.036986       0.0     0.0           0.0  ...    0.0   
2       0.0       0.0   0.000000       0.0     0.0           0.0  ...    0.0   
3       0.0       0.0   0.000000       0.0     0.0           0.0  ...    0.0   
4       0.0       0.0   0.000000       0.0     0.0           0.0  ...    0.0   

   южный   явиться   явление      язык  январь  япония  